# Graph of Thought

PES2UG23CS928

In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import networkx as nx

In [2]:
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [3]:
def generate_text(prompt, max_new_tokens=80):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [4]:
def generate_thoughts(question, k=3):
    thoughts = []
    for _ in range(k):
        prompt = f"""
        Question: {question}
        Think step by step and suggest one possible reasoning path.
        Thought:
        """
        thought = generate_text(prompt)
        thoughts.append(thought)
    return thoughts

In [5]:
def expand_thought(thought):
    prompt = f"""
    Here is a reasoning step:
    {thought}
    Continue this reasoning logically:
    """
    return generate_text(prompt)

In [6]:
def score_thought(thought):
    score = len(thought)
    logic_keywords = ["because", "therefore", "so", "hence", "thus"]
    for word in logic_keywords:
        if word in thought.lower():
            score += 20
    return score

In [7]:
def graph_of_thought_solver(question, branches=3, depth=2):
    print("Generating initial thoughts...\n")

    G = nx.DiGraph()
    node_id = 0

    G.add_node(node_id, content=question, score=0)
    root_id = node_id
    node_id += 1

    initial_thoughts = generate_thoughts(question, branches)
    level_nodes = []

    for thought in initial_thoughts:
        score = score_thought(thought)
        G.add_node(node_id, content=thought, score=score)
        G.add_edge(root_id, node_id)
        level_nodes.append(node_id)
        node_id += 1

    for _ in range(depth):
        new_level = []
        for parent in level_nodes:
            parent_thought = G.nodes[parent]['content']
            expanded = expand_thought(parent_thought)
            score = score_thought(expanded)

            G.add_node(node_id, content=expanded, score=score)
            G.add_edge(parent, node_id)

            if len(G.nodes) > 2:
                merge_target = list(G.nodes)[-2]
                if merge_target != parent:
                    G.add_edge(merge_target, node_id)

            new_level.append(node_id)
            node_id += 1

        level_nodes = new_level

    for node in nx.topological_sort(G):
        for parent in G.predecessors(node):
            G.nodes[node]['score'] += 0.5 * G.nodes[parent]['score']

    best_node = max(G.nodes, key=lambda n: G.nodes[n]['score'])
    best_path = nx.shortest_path(G, source=root_id, target=best_node)

    best_reasoning = "\n".join([G.nodes[n]['content'] for n in best_path])

    final_prompt = f"""
    Based on the reasoning below, give the final answer clearly.
    Reasoning:
    {best_reasoning}
    Final Answer:
    """
    final_answer = generate_text(final_prompt)

    return final_answer, best_reasoning, G

In [8]:
question = "If a train travels 60 km in 1 hour and then 30 km in 30 minutes, what is its average speed?"

answer, reasoning_path, graph = graph_of_thought_solver(question)

print("\nBest Reasoning Path:\n", reasoning_path)
print("\nFinal Answer:\n", answer)

Generating initial thoughts...


Best Reasoning Path:
 If a train travels 60 km in 1 hour and then 30 km in 30 minutes, what is its average speed?
For 1800km.for 680 min', 6hour period * 3=800 miles Total tom takes to cross 5k tracks or 200 = 2160 min for this interval a fast, 380 mi per 255minute trip = 1600/10800 of average acceleration to reach 10 metres distance train and also trains from 480mile in half hours Speed rate between 1000ps 200
180600 = 28120 x 1000p * 120h/480 + 40gmpr* 3250s+490m= 8750s at 40minute interval Fast times 1800p/ (x hours time = 11240pt/2400v+2820v/10850vest time > 1800 metres' THREPAENT, MAIL POLE. Speed T&G
The same time in 1/4 of 30000, and we will be paying 190700*. = 120400 hours Fast times 10240b Time 10= 1800 times 1200= 1300 + 100/2= 1600 Speed and it add this in that there for every 1000 km with its Time or Average The same as 106000 days at 1500 sec interval with Time= (Locate Time +

Final Answer:
 Each 30 sec period from hour * minutes > maximu